# LLM-Wiki

Arbeitsnotebook zu [Anforderungen.md](Anforderungen.md). Begriffe stehen im [Glossar](../README.md#glossar).

Stellen mit `TODO` ergänzt ihr selbst. Zellen **der Reihe nach** ausführen.

# Imports

In [ ]:
import os
import re
from datetime import datetime
from pathlib import Path

# Anfragen an Ollama auf dem eigenen Rechner sollen nicht über einen Proxy laufen
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
# Kennung, mit der Websites abgerufen werden
os.environ["USER_AGENT"] = "ausbildung-uebungen"

from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

# LLM-Initialisierung

In [ ]:
llm = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url="http://localhost:11434",
    temperature=0,
)

print(llm.invoke("Sag kurz Hallo.").content)

# Phase 1: Das Wiki-Gerüst

### Hilfsfunktionen (vorgegeben)

Diese Funktionen sind fertig. Lest sie durch, bevor ihr weitermacht: Ihr müsst verstehen, wie eine Seite aufgebaut ist.

Jede Seite hat dieses Format:

```markdown
# Titel

> Zusammenfassung in einem Satz.

Inhalt mit Verweisen wie [[Andere Seite]].

Quellen: ...
```

In [ ]:
WIKI = Path("wiki")
QUELLEN = Path("quellen")
WIKI.mkdir(exist_ok=True)
QUELLEN.mkdir(exist_ok=True)

SONDERSEITEN = {"index.md", "log.md"}


def dateiname(titel):
    """'Regeln für Anfänger' -> wiki/regeln-fuer-anfaenger.md"""
    name = titel.lower()
    for alt, neu in {"ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"}.items():
        name = name.replace(alt, neu)
    name = re.sub(r"[^a-z0-9]+", "-", name).strip("-")
    return WIKI / f"{name}.md"


def seite_schreiben(titel, zusammenfassung, inhalt):
    """Legt eine Seite im festen Format an oder überschreibt sie."""
    text = f"# {titel}\n\n> {zusammenfassung}\n\n{inhalt.strip()}\n"
    dateiname(titel).write_text(text, encoding="utf-8")


def seite_lesen(titel):
    """Gibt den Text einer Seite zurück oder None, wenn es sie nicht gibt."""
    pfad = dateiname(titel)
    return pfad.read_text(encoding="utf-8") if pfad.exists() else None


def kopf_lesen(pfad):
    """Liest Titel (erste Zeile) und Zusammenfassung (Zeile mit '> ') einer Seite."""
    zeilen = pfad.read_text(encoding="utf-8").splitlines()
    titel = zeilen[0].removeprefix("# ").strip()
    zusammenfassung = next((z.removeprefix("> ").strip() for z in zeilen if z.startswith("> ")), "")
    return titel, zusammenfassung


def alle_seiten():
    """Titel aller Wiki-Seiten, alphabetisch."""
    return sorted(kopf_lesen(p)[0] for p in WIKI.glob("*.md") if p.name not in SONDERSEITEN)


def index_aktualisieren():
    """Schreibt wiki/index.md neu: eine Zeile pro Seite mit Titel und Zusammenfassung."""
    zeilen = ["# Index", ""]
    for pfad in sorted(WIKI.glob("*.md")):
        if pfad.name not in SONDERSEITEN:
            titel, zusammenfassung = kopf_lesen(pfad)
            zeilen.append(f"- [[{titel}]]: {zusammenfassung}")
    (WIKI / "index.md").write_text("\n".join(zeilen) + "\n", encoding="utf-8")


def log(eintrag):
    """Hängt eine Zeile mit Zeitstempel an wiki/log.md an."""
    with open(WIKI / "log.md", "a", encoding="utf-8") as datei:
        datei.write(f"- {datetime.now():%Y-%m-%d %H:%M} {eintrag}\n")

### Übung: Drei Seiten von Hand anlegen

Legt mit `seite_schreiben` drei Seiten an, die aufeinander verweisen, und ruft danach `index_aktualisieren()` und `log(...)` auf. Schaut euch dann `wiki/index.md` und `wiki/log.md` an.

In [ ]:
# TODO: seite_schreiben(...) dreimal aufrufen

index_aktualisieren()
print((WIKI / "index.md").read_text(encoding="utf-8"))

### Übung: Links finden

Schreibt eine Funktion, die alle `[[Links]]` aus einem Text findet. Ihr braucht sie in Phase 5.

`links_finden("Siehe [[Alex]] und [[Spielregeln]].")` soll `["Alex", "Spielregeln"]` liefern.

*Tipp: `re.findall(r"\[\[(.+?)\]\]", text)`*

In [ ]:
def links_finden(text):
    # TODO
    return []


print(links_finden("Siehe [[Alex]] und [[Spielregeln]]."))

# Phase 2: Merken

### Datenmodelle für die strukturierte Ausgabe (vorgegeben)

Mit Pydantic beschreibt ihr, in welcher Form das LLM antworten soll. Die `description` jedes Feldes wird dem LLM mitgeschickt, sie ist also Teil des Prompts.

In [ ]:
class WikiSeite(BaseModel):
    titel: str = Field(description="Kurzer, eindeutiger Titel, z. B. ein Name oder ein Begriff")
    zusammenfassung: str = Field(description="Ein Satz, worum es auf der Seite geht")
    inhalt: str = Field(description="Inhalt in Markdown. Verweise auf andere Seiten als [[Titel]]. Am Ende eine Zeile 'Quellen: ...'")


class WikiAenderung(BaseModel):
    seiten: list[WikiSeite] = Field(description="Neue Seiten oder bestehende Seiten mit vollständigem, ergänztem Inhalt")


class Seitenauswahl(BaseModel):
    titel: list[str] = Field(description="Titel der passenden Seiten aus dem Index, höchstens 5")


REGELN = """Du pflegst ein Wiki aus Markdown-Seiten.
Regeln:
- Eine Seite pro Person, Thema oder Begriff.
- Übernimm nur Fakten aus dem gegebenen Text. Erfinde nichts.
- Gibt es schon eine Seite zum selben Thema, ergänze sie, statt eine neue anzulegen.
  Gib dann den vollständigen neuen Inhalt der Seite zurück, nicht nur die Änderung.
- Verweise auf andere Seiten mit [[Titel]].
- Nenne am Ende jeder Seite die Quelle in einer Zeile 'Quellen: ...'.
"""

### Test: Funktioniert die strukturierte Ausgabe?

`with_structured_output` sorgt dafür, dass das LLM ein `WikiAenderung`-Objekt zurückgibt statt freiem Text.

Gibt es hier einen Fehler, probiert `llm.with_structured_output(WikiAenderung, method="json_schema")`.

In [ ]:
llm_wiki = llm.with_structured_output(WikiAenderung)

ergebnis = llm_wiki.invoke(REGELN + "\nText: Kim spielt gern Schach und wohnt in Bremen.\nQuelle: Test")
print(ergebnis)

### Funktion `seiten_auswaehlen(text)`

Das LLM bekommt den Index und wählt die Seiten aus, die zum Text passen. Diese Funktion braucht ihr beim Merken **und** beim Fragen.

In [ ]:
def seiten_auswaehlen(text):
    if not alle_seiten():
        return []
    index = (WIKI / "index.md").read_text(encoding="utf-8")
    # TODO: llm.with_structured_output(Seitenauswahl) mit Index und Text aufrufen
    #       und die Liste der Titel zurückgeben (nur Titel, die es wirklich gibt)
    return []

### Funktion `merken(text, quelle)`

1. Passende vorhandene Seiten mit `seiten_auswaehlen` finden und deren Inhalt lesen
2. `REGELN`, vorhandene Seiten, neuen Text und Quelle an `llm_wiki` geben
3. Jede zurückgegebene Seite mit `seite_schreiben` speichern
4. `index_aktualisieren()` und `log(...)` aufrufen

In [ ]:
def merken(text, quelle="Chat"):
    # TODO
    pass

### Test: Merken und Ergänzen

In [ ]:
# merken("Ich heiße Alex und mache eine Ausbildung in der Anwendungsentwicklung.")
# merken("Alex spielt in der Freizeit Volleyball.")
# print(alle_seiten())

# Phase 3: Quellen einlesen

Ladet eure eigenen Quellen aus Projekt 1 (siehe `../projekt-1-rag/quellen.md`), speichert den Text in `quellen/` und lasst das LLM daraus Wiki-Seiten erstellen.

Lange Texte passen nicht auf einmal in den Prompt. Teilt sie in Stücke von etwa 3000 Zeichen und ruft `merken` für jedes Stück auf.

In [ ]:
pdf_dateien = sorted(Path("../projekt-1-rag/data").glob("*.pdf"))
print([p.name for p in pdf_dateien])

# TODO: für jedes PDF den Text laden, in quellen/<name>.txt speichern
#       und stückweise mit merken(stueck, quelle=pdf.name) ins Wiki übernehmen

In [ ]:
WEBSITES = [
    # TODO: eure URLs aus Projekt 1
]

# TODO: wie oben, mit WebBaseLoader

# Phase 4: Fragen beantworten

### Funktion `fragen(frage)`

1. Mit `seiten_auswaehlen` die passenden Seiten finden
2. Seiten lesen und zusammen mit der Frage an das LLM geben
3. Antwort mit Angabe der genutzten Seiten ausgeben oder „Das weiß ich nicht“, wenn keine Seite passt

In [ ]:
def fragen(frage):
    # TODO
    return "TODO"

### Test

Fragt auch nach etwas, das ihr dem Wiki **nie** erzählt habt. Das LLM soll dann zugeben, dass es die Antwort nicht weiß.

In [ ]:
# Eure Testfragen aus ../projekt-1-rag/testfragen.md
# print(fragen("..."))
# print(fragen("Wie heiße ich?"))
# print(fragen("Wie heißt mein Haustier?"))  # nie erzählt!

# Phase 5 (Bonus): Aufräumen, Chat, Vergleich

Siehe [Anforderungen.md](Anforderungen.md#phase-5-bonus).

In [ ]:
# Platz für eure Erweiterungen